## client HTTP/REST en Python avec requests

In [ ]:
# ------------------------ utiliser les fonctions globales (get(), post(), ...) ---------------

import requests

URL = "https://gorest.co.in/public/v3/users"

# requests.(<get>,<post>,<put>,<patch>,<delete>,<head> on la même interface !!! params, data, files, headers, url, ...)
r = requests.get(URL, params={"page[size]": 5}, timeout=10, verify=False)

print(r.status_code)                    # 200
# CaseInsensitiveDict génère des clés uniques mais insensibles à la casse ==> "Content-Type" == "content-type"
print(r.headers["Content-Type"])        # application/json...
print(r.url)                            # URL réellement envoyée
print(r.elapsed.total_seconds())        # durée de l'échange

# En HTTP/REST le corpts de réponse devrait être soit application/json ou application/xml à la rigueur
if "application/json" in r.headers["Content-Type"]:
    # transforme le corps convertible en json étant donné le header , en list[dict] ou dict
    corps = r.json()
    for u in corps["data"]:
        print(u["id"], u["name"], u["status"])
    print(corps["meta"])                    # has_more, next_cursor, total

### alternatives
# si content-type == text/html ou application/xml ou text/plain ==> r.text ( str CHARSET Utf-8)
# si content-type == application/zip, video/mp4, application/octet-stream ==> r.content ( bytes )


In [ ]:
# -------------------------- RAPPEL sur getattr,  setattr, delattr, hasattr ----------

class Truc:
    pass

t = Truc()
print("attribut param ? ", hasattr(t, "param"))
setattr(t, "param", 10)
print("attribut param ? ", hasattr(t, "param"))
print(getattr(t, "param"))
delattr(t, "param")
print("attribut param ? ", hasattr(t, "param"))

print("-"*10 + "avec n'importe quoi objet comme requests" + "-"*10)
method = input("méthode http ?")
# méthode dynamique
getattr(requests, method)


In [ ]:
# ---------------------------------- idem avec la classe de base Request -----------------------
import requests
import urllib3 # dépendance tierce utilisée par requests

urllib3.disable_warnings() # enlever les warnings du verify

## REM: Request a la même signature que les fonctions globales mais rajoute la méthode

req = requests.Request(
    "GET",
    "https://gorest.co.in/public/v3/users",
    params={"filter[status]": "active", "sort": "name", "page[size]": 3},
    headers={"Accept": "application/json"},
)
prepared = req.prepare()

print(prepared.method, prepared.url)
print(dict(prepared.headers))
print(prepared.body)           # None pour un GET

##### pour envoyer la requête préparée : on doit créer un objet requests.Session(...)
s = requests.Session()
### ... envoyer la requête préparée avec .send()
r = s.send(prepared, allow_redirects=False, verify=False)

# if r.ok
if 200 <= r.status_code < 300:
    if "application/json" in r.headers["Content-Type"]:
        corps = r.json()
        for u in corps["data"]:
            print(u["id"], u["name"], u["status"])
        print(corps["meta"])
    else:
        print(f"mauvais content-type {r.headers["Content-Type"]} !")
else:
    print(f"mauvais status {r.status_code} !")



In [ ]:
# ------------------------ sécuriser les paramètres (injections) --------------------
from urllib.parse import quote # urllib est auto inclus avec python

# FIXES: protocole, domaine, radical, version
BASE = "https://gorest.co.in/public/v3"

# ici on fixe la ressource mais on pourrait aussi la paramétrer

def url_utilisateur(user_id: str) -> str:
    """
    retourne l'url d'un endpoint en utilisant quote qui peut échapper les paramètres de chemin ou de requêtes
    ICI: endpoint pour UN SEUL id => pas de query params => safe="" neutralise tout
    """
    return f"{BASE}/users/{quote(user_id, safe='')}"


# injection possible: /
r = requests.get(url=BASE + '/019d00aa/b240-7d95-8cc2-479f04f365d7')
print(r.url, r.json())

# / urlencodé donc neutralisé
url = url_utilisateur('019d00aa/b240-7d95-8cc2-479f04f365d7')
print(url)
r = requests.get(url=url)
print(r.url, r.json())


In [ ]:
# ------------------- inejections de configuration avec @dataclass et load_dotenv --------------
from dotenv import load_dotenv
import os
from dataclasses import dataclass, field

load_dotenv()

# print(os.getenv('GOREST_BASE_URL'), os.getenv('GOREST_TOKEN'))

# décorateur paramétré qui tranforme la classe en objet de données en ajoutant un __init__ mais customisable !!
# ce __init__ nécessite des annotations (contrôle de type !!)
@dataclass(
    frozen=True,  # ce __init__ retourne un objet immuable
    kw_only=True # les attributs ne peuvent appelés que de façon nommée (*, param="xx")
)
class Settings:
    base_url: str = ""
    token: str | None = field(default=None, repr=False) # repr: non lisible dans les logs !!
    timeout: tuple[float, float] = (3.05, 10.0) # (connexion, lecture) en secondes

    @classmethod
    def form_env(cls):
        """méthode de classe à utilier directement avec Settings.from_env(...)"""
        return cls(
            base_url=os.getenv('GOREST_BASE_URL'),
            token=os.getenv('GOREST_TOKEN') or None
        )

obj = Settings.form_env()
print(obj) # on ne voit pas le secret
# obj.token = ... # ERROR

In [ ]:
# ---------------- gestion d'entête Bearer côté client et réutilisable!!
from dotenv import load_dotenv
import os
import requests


load_dotenv()

token = os.getenv("GOREST_TOKEN")

# 1. En-tête à la main
# r = requests.get(URL, headers={"Authorization": f"Bearer {token}"}, timeout=10)

#### 2. Objet d'authentification réutilisable
class BearerAuth(requests.auth.AuthBase):
    def __init__(self, token: str):
        self.token = token

    def __call__(self, r: requests.PreparedRequest) -> requests.PreparedRequest:
        """
        l'objet BearAuth(token) doit être lui même appelable !!!
        auth = BearAuth(token)
        auth(...)
        """
        r.headers["Authorization"] = f"Bearer {self.token}"
        return r

URL = "https://gorest.co.in/public/v3/users"

# r = requests.post(
#     URL,
#     auth=BearerAuth(token), 
#     json={"name":"matt2 LAMAM2","email":"mlamamra2@example.com"}
# )
# print(r.status_code, r.json())

# # les données crées sont privées donc le GET demande aussi le jeton !!
# r = requests.get(
#     f'{URL}/{r.json()["id"]}', 
#     auth=BearerAuth(token)
# )
# print(r.status_code, r.json())

### 3. Session : l'authentification s'applique à tous les appels
# s = requests.Session()

# les sessions utilisent les gestionnaires de contexte
with requests.Session() as s:
    s.auth = BearerAuth(token)

    r = s.post(URL, timeout=10)
    r = s.get(f"{URL}/{r.json()["id"]}", timeout=10)s
# s.close()



201 {'created_at': '2026-11-16T16:35:02.075298Z', 'email': 'mlamamra2@example.com', 'id': '01a1e536-473c-728d-af22-6bdebe868f48', 'name': 'matt2 LAMAM2', 'read_only': False, 'status': 'active', 'updated_at': '2026-11-16T16:35:02.075298Z'}
200 {'created_at': '2026-11-16T16:35:02.075298Z', 'email': 'mlamamra2@example.com', 'id': '01a1e536-473c-728d-af22-6bdebe868f48', 'name': 'matt2 LAMAM2', 'read_only': False, 'status': 'active', 'updated_at': '2026-11-16T16:35:02.075298Z'}
